# Preprocess Only - Batch Processing

**Process labeled VifactCheck data with memory-efficient batch processing**

Input: `./data/json/labeled_nei_as_real/news_data_vifactcheck_*_labeled.json`
Output: `./processed_data/hdf5/vifactcheck_{train,dev,test}.h5`


In [4]:
# Install dependencies
import subprocess
import sys

packages = [
    "pandas",
    "numpy",
    "torch",
    "torchvision",
    "transformers",
    "pillow",
    "matplotlib",
    "tqdm",
]

for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"✅ {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"📦 Installed {pkg}")

✅ pandas
✅ numpy
✅ torch
✅ torchvision


/opt/miniconda3/envs/fake_news/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ transformers
📦 Installed pillow
✅ matplotlib
✅ tqdm


In [5]:
# Setup
import sys
import os
import orjson
import numpy as np
import torch
import gc
from pathlib import Path
from tqdm import tqdm

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"✅ Ready: {project_root}")

✅ Ready: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-new-detection


In [ ]:
# Configuration
PREPROCESSING_CONFIG = {
    # --- Text / Image features (Stage 1 - COOLANT retraining) ---
    "text_model": "vinai/phobert-base-v2",  # standardized across Stage 1 & 2
    "image_model": "resnet50",
    "language": "vi",
    "max_length": 512,
    "batch_size": 32,
    "use_word_segmentation": True,  # underthesea word_tokenize for PhoBERT
    "input_dir": "./data/json/labeled_nei_as_real",
    "output_dir": "./processed_data/hdf5",
    # --- Evidence retrieval (Stage 0 - for MM-ViFactCheck Stage 2) ---
    "retrieval_top_n": 5,
    "evidence_output_dir": "./processed_data/evidence",
    "vifactcheck_dataset": "tranthaihoa/vifactcheck",
    "vifactcheck_splits": ["train", "test", "dev"],
}

import os, torch

os.makedirs(PREPROCESSING_CONFIG["output_dir"], exist_ok=True)
os.makedirs(PREPROCESSING_CONFIG["evidence_output_dir"], exist_ok=True)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Device: {device}")
print(f"Word segmentation: {PREPROCESSING_CONFIG['use_word_segmentation']}")
print(f"Text model: {PREPROCESSING_CONFIG['text_model']}")

## Stage 0 — Evidence Retrieval (SBERT + BM25)

Load ViFactCheck from HuggingFace, run hybrid SBERT+BM25 retrieval on the Context field,
and save `evidence_top5` per sample to JSON.

These JSONs are used by the **Stage 2 MM-ViFactCheck** training notebook — they are independent of
the COOLANT HDF5 feature files produced below.

> **Skip this section** if `processed_data/evidence/` already exists and you only need to
> re-extract COOLANT features.


In [ ]:
# Stage 0a — Load ViFactCheck from HuggingFace
from datasets import load_dataset

vifactcheck_splits = {}
for split in PREPROCESSING_CONFIG["vifactcheck_splits"]:
    ds = load_dataset(PREPROCESSING_CONFIG["vifactcheck_dataset"], split=split)
    records = ds.to_list()
    vifactcheck_splits[split] = records
    print(f"  {split}: {len(records)} samples, columns: {list(records[0].keys())}")

print(f"\nLoaded {sum(len(v) for v in vifactcheck_splits.values())} total samples")

In [ ]:
# Stage 0b — SBERT+BM25 evidence retrieval → save evidence JSONs
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str(Path("../src").resolve()))

from src.preprocessing.evidence_retrieval import (
    EvidenceRetriever,
    save_evidence,
    load_evidence,
)

evidence_dir = Path(PREPROCESSING_CONFIG["evidence_output_dir"])

retriever = EvidenceRetriever(device=device)
print(f"EvidenceRetriever ready on {retriever.device}")

for split, records in vifactcheck_splits.items():
    out_path = evidence_dir / f"evidence_{split}.json"

    if out_path.exists():
        print(f"  {split}: already exists at {out_path}, skipping")
        continue

    print(f"\n--- Retrieving evidence for {split} ({len(records)} samples) ---")
    enriched = retriever.process_split(
        records,
        statement_key="Statement",
        context_key="Context",
        top_n=PREPROCESSING_CONFIG["retrieval_top_n"],
    )
    save_evidence(enriched, str(out_path))

print("\nEvidence retrieval complete.")
for split in PREPROCESSING_CONFIG["vifactcheck_splits"]:
    p = evidence_dir / f"evidence_{split}.json"
    if p.exists():
        sample = load_evidence(str(p))[0]
        print(
            f"  {split}: {p.name}  |  evidence_top5 sample: {sample.get('evidence_top5', [])[:1]}"
        )

## Stage 1 — Feature Extraction for COOLANT Retraining

Extract PhoBERT-base-v2 token embeddings (with VnCoreNLP/underthesea word segmentation)
and ResNet50 image features. Output: per-split HDF5 files for COOLANT training.

> **Re-run this section** any time you want to retrain COOLANT from scratch or with different
> backbone settings. The evidence JSONs above are unaffected.


In [ ]:
# Initialize preprocessor (Stage 1 — COOLANT feature extraction)
try:
    from src.preprocessing import CombinedPreprocessor

    preprocessor = CombinedPreprocessor(
        text_model_name=PREPROCESSING_CONFIG["text_model"],
        image_model_name=PREPROCESSING_CONFIG["image_model"],
        language=PREPROCESSING_CONFIG["language"],
        max_text_length=PREPROCESSING_CONFIG["max_length"],
        device=device,
        use_word_segmentation=PREPROCESSING_CONFIG["use_word_segmentation"],
    )
    seg_status = (
        "underthesea"
        if preprocessor.text_preprocessor.use_word_segmentation
        else "disabled"
    )
    print(f"✅ Preprocessor ready  |  word segmentation: {seg_status}")
except Exception as e:
    print(f"❌ Preprocessor error: {e}")

/opt/miniconda3/envs/fake_news/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/miniconda3/envs/fake_news/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/haila/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 86.3MB/s]


✅ Preprocessor ready


In [ ]:
# Load labeled data - process each split separately
from collections import Counter


def load_split(split_name):
    """Load a single labeled split (train/dev/test)."""
    data_dir = Path(PREPROCESSING_CONFIG["input_dir"])
    json_file = data_dir / f"news_data_vifactcheck_{split_name}_labeled.json"

    if not json_file.exists():
        print(f"File not found: {json_file}")
        return []

    with open(json_file, "rb") as f:
        data = orjson.loads(f.read())
    print(f"Loaded {len(data)} from {json_file.name}")
    return data


# Load each split separately to preserve original ViFactCheck splits
splits = {}
for split_name in ["train", "dev", "test"]:
    articles = load_split(split_name)
    splits[split_name] = articles

    # Verify labels exist
    labels_in_split = [a.get("label") for a in articles]
    label_dist = Counter(labels_in_split)
    print(f"  {split_name}: {len(articles)} samples, labels={dict(label_dist)}")
    assert None not in label_dist, f"ERROR: {split_name} has records without labels!"
    assert (
        len(label_dist) >= 2
    ), f"ERROR: {split_name} has only {len(label_dist)} class(es)!"

print(f"\nTotal: {sum(len(v) for v in splits.values())} samples across 3 splits")

In [ ]:
# Prepare data for each split
def prepare_split(articles):
    """Extract texts, image paths, and labels from articles."""
    texts = []
    image_paths = []
    labels = []

    for article in articles:
        text = (
            article.get("text", "")
            or article.get("content", "")
            or article.get("title", "")
        )
        if not text.strip():
            continue

        texts.append(text.strip())
        images = article.get("images", [])

        # Extract image path
        if images and len(images) > 0:
            if isinstance(images[0], str):
                img_path = images[0]
            elif isinstance(images[0], dict):
                img_path = (
                    images[0].get("folder_path", "")
                    or images[0].get("url", "")
                    or images[0].get("path", "")
                )
            else:
                img_path = str(images[0])
            image_paths.append(img_path if img_path else None)
        else:
            image_paths.append(None)

        # Extract label - must exist in labeled data
        label = article["label"]  # KeyError if missing = good, catches bugs
        if isinstance(label, str):
            label = 1 if label.lower() in ["fake", "false", "1"] else 0
        labels.append(int(label))

    return texts, image_paths, labels


split_data = {}
for split_name, articles in splits.items():
    texts, image_paths, labels = prepare_split(articles)
    split_data[split_name] = {
        "texts": texts,
        "image_paths": image_paths,
        "labels": labels,
    }
    label_dist = Counter(labels)
    n_with_images = sum(1 for p in image_paths if p is not None)
    print(
        f"{split_name}: {len(texts)} samples, labels={dict(label_dist)}, with_images={n_with_images}"
    )

In [ ]:
# Create placeholder images - use ZERO images (not random noise)
# Random noise creates meaningless ResNet features that degrade training
from PIL import Image

placeholder_dir = Path("./placeholder_images")
placeholder_dir.mkdir(exist_ok=True)

# Create a single zero placeholder image (black image)
zero_placeholder_path = placeholder_dir / "zero_placeholder.jpg"
if not zero_placeholder_path.exists():
    zero_array = np.zeros((224, 224, 3), dtype=np.uint8)
    Image.fromarray(zero_array).save(zero_placeholder_path)

for split_name, data in split_data.items():
    processed_paths = []
    n_missing = 0
    for img_path in data["image_paths"]:
        if not img_path or not os.path.exists(img_path):
            processed_paths.append(str(zero_placeholder_path))
            n_missing += 1
        else:
            processed_paths.append(img_path)
    data["processed_image_paths"] = processed_paths
    print(
        f"{split_name}: {n_missing}/{len(processed_paths)} images replaced with zero placeholder"
    )

In [ ]:
# Batch preprocessing - process each split separately
BATCH_SIZE = PREPROCESSING_CONFIG["batch_size"]

for split_name, data in split_data.items():
    texts = data["texts"]
    processed_paths = data["processed_image_paths"]
    labels = data["labels"]

    total_samples = len(texts)
    num_batches = (total_samples + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"\nProcessing {split_name}: {total_samples} samples, {num_batches} batches")

    all_text_features = []
    all_image_features = []

    for batch_idx in tqdm(range(num_batches), desc=f"Processing {split_name}"):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min((batch_idx + 1) * BATCH_SIZE, total_samples)

        batch_texts = texts[start_idx:end_idx]
        batch_image_paths = processed_paths[start_idx:end_idx]

        text_features = preprocessor.text_preprocessor.extract_token_embeddings(
            batch_texts
        )
        image_features = preprocessor.image_preprocessor.extract_features(
            batch_image_paths
        )

        all_text_features.append(text_features)
        all_image_features.append(image_features)

        del text_features, image_features
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif torch.backends.mps.is_available():
            torch.mps.empty_cache()

    data["text_features"] = np.vstack(all_text_features)
    data["image_features"] = np.vstack(all_image_features)
    data["labels_array"] = np.array(labels)

    print(
        f"  Text: {data['text_features'].shape}, Image: {data['image_features'].shape}"
    )
    print(f"  Labels: {Counter(labels)}")

    del all_text_features, all_image_features
    gc.collect()

print("\nAll splits processed.")

In [ ]:
# Save each split as a separate HDF5 file
import h5py

output_dir = Path(PREPROCESSING_CONFIG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

for split_name, data in split_data.items():
    text_features = data["text_features"]
    image_features = data["image_features"]
    labels = data["labels_array"]
    n_samples = len(labels)

    hdf5_path = output_dir / f"vifactcheck_{split_name}.h5"
    print(f"\nSaving {split_name}: {n_samples} samples -> {hdf5_path}")

    chunk_size = min(64, n_samples)

    with h5py.File(hdf5_path, "w") as f:
        f.create_dataset(
            "text_features",
            data=text_features,
            chunks=(chunk_size, text_features.shape[1], text_features.shape[2]),
            compression="gzip",
            compression_opts=4,
        )
        f.create_dataset(
            "image_features",
            data=image_features,
            chunks=(chunk_size, image_features.shape[1]),
            compression="gzip",
            compression_opts=4,
        )
        f.create_dataset("labels", data=labels)

        # Store metadata
        f.attrs["n_samples"] = n_samples
        f.attrs["text_shape"] = text_features.shape[1:]
        f.attrs["image_shape"] = image_features.shape[1:]
        f.attrs["label_distribution"] = str(dict(Counter(labels.tolist())))

    size_mb = hdf5_path.stat().st_size / (1024 * 1024)
    print(f"  Shape: text={text_features.shape}, image={image_features.shape}")
    print(f"  Labels: {dict(Counter(labels.tolist()))}")
    print(f"  Size: {size_mb:.1f} MB")

print("\nAll HDF5 files saved.")

In [ ]:
# Save metadata
import json

output_dir = Path(PREPROCESSING_CONFIG["output_dir"])
metadata = {}
for split_name, data in split_data.items():
    metadata[split_name] = {
        "num_samples": len(data["labels"]),
        "text_feature_shape": list(data["text_features"].shape),
        "image_feature_shape": list(data["image_features"].shape),
        "label_distribution": dict(Counter(data["labels"])),
    }

metadata["config"] = PREPROCESSING_CONFIG

with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved:")
for split_name, info in metadata.items():
    if split_name != "config":
        print(
            f"  {split_name}: {info['num_samples']} samples, labels={info['label_distribution']}"
        )

# Cleanup
for data in split_data.values():
    if "text_features" in data:
        del data["text_features"]
    if "image_features" in data:
        del data["image_features"]
gc.collect()

In [ ]:
# Final status
print("Preprocessing Complete!")
print("=" * 50)

output_dir = Path(PREPROCESSING_CONFIG["output_dir"])
if output_dir.exists():
    for f in sorted(output_dir.glob("*")):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name}: {size_mb:.2f} MB")

print(f"\nNext step: Run train_vietnamese_coolant.ipynb")
print(f"HDF5 dir: {output_dir}")